<a href="https://colab.research.google.com/github/mohmadmansour54-ship-it/Cloud-Lab2/blob/main/students_form.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 טופס סטודנטים – students.json
1. הריצו את תא החיבור לדרייב.
2. אם `students.json` עוד לא בדרייב – הריצו את תא 2 (יוצר אותו). עדכנו בו את פרטי הקבוצה האמיתיים.
3. הריצו את תא הטופס.

In [1]:
# תא 1 – חיבור ל-Google Drive
from google.colab import drive
drive.mount('/content/drive')

JSON_PATH = '/content/drive/MyDrive/students.json'

Mounted at /content/drive


In [2]:
# תא 2 – יצירת students.json בדרייב (רק אם הוא לא קיים)
import json, os

students_data = [
    {
        "id": 1,
        "first_name": "Muhamad",
        "last_name": "LAST_NAME",
        "email": "muhamad.xxx@e.braude.ac.il",
        "courses": [
            "Automata and Computability",
            "Machine Learning",
            "AI Engineering"
        ],
        "interesting_link": "https://colab.research.google.com",
        "favorite_show": ""
    },
    {
        "id": 2,
        "first_name": "Student2",
        "last_name": "LAST_NAME",
        "email": "student2@e.braude.ac.il",
        "courses": [
            "Course A",
            "Course B"
        ],
        "interesting_link": "https://github.com",
        "favorite_show": ""
    },
    {
        "id": 3,
        "first_name": "Student3",
        "last_name": "LAST_NAME",
        "email": "student3@e.braude.ac.il",
        "courses": [
            "Course A",
            "Course C"
        ],
        "interesting_link": "https://stackoverflow.com",
        "favorite_show": ""
    }
]

if not os.path.exists(JSON_PATH):
    with open(JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(students_data, f, ensure_ascii=False, indent=2)
    print('✅ students.json נוצר ב-Drive')
else:
    print('ℹ️ הקובץ כבר קיים – לא נדרס')

ℹ️ הקובץ כבר קיים – לא נדרס


In [3]:
# תא 3 – פונקציות קריאה/כתיבה
import json

def load_students():
    with open(JSON_PATH, encoding='utf-8') as f:
        return json.load(f)

def save_students(students):
    with open(JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(students, f, ensure_ascii=False, indent=2)

def update_favorite_show(student_id, show):
    students = load_students()          # קוראים מחדש כדי לא לדרוס שינויים
    for s in students:
        if s['id'] == student_id:
            s['favorite_show'] = show
            save_students(students)
            return s
    raise ValueError(f'סטודנט {student_id} לא נמצא')

In [4]:
# תא 4 – הטופס 🎨
import ipywidgets as widgets
from IPython.display import display, HTML
from html import escape

display(HTML('''
<style>
.card {
  direction: rtl; max-width: 560px; padding: 22px 26px; border-radius: 18px;
  background: linear-gradient(135deg, #fdfbfb 0%, #ebedee 100%);
  box-shadow: 0 8px 24px rgba(60,60,120,.18); border-top: 6px solid #6c5ce7;
}
.card-title { font: 700 22px/1.3 "Segoe UI", sans-serif; color: #4834d4; margin-bottom: 6px; }
.card-sub   { font: 13px "Segoe UI", sans-serif; color: #636e72; margin-bottom: 14px; }
.card .widget-label { font-weight: 600; color: #2d3436; }
.card input, .card textarea, .card select { border-radius: 8px !important; }
.fav-box input { background: #fff8e1 !important; border: 2px solid #fdcb6e !important; }
.link-box a { color: #0984e3; font-weight: 600; text-decoration: none; }
</style>'''))

style  = {'description_width': '120px'}
wide   = widgets.Layout(width='500px')

title = widgets.HTML('<div class="card-title">📋 כרטיס סטודנט</div>'
                     '<div class="card-sub">בחרו סטודנט כדי לראות ולעדכן את הפרטים</div>')

students = load_students()
dropdown = widgets.Dropdown(
    options=[('— בחרו סטודנט —', None)] +
            [(f"{s['first_name']} {s['last_name']}", s['id']) for s in students],
    description='סטודנט:', style=style, layout=wide)

first_name = widgets.Text(description='שם פרטי:',  disabled=True, style=style, layout=wide)
last_name  = widgets.Text(description='שם משפחה:', disabled=True, style=style, layout=wide)
email      = widgets.Text(description='מייל בראודה:', disabled=True, style=style, layout=wide)
courses    = widgets.Textarea(description='קורסים:', disabled=True, style=style,
                              layout=widgets.Layout(width='500px', height='100px'))
link       = widgets.HTML()
link.add_class('link-box')

fav = widgets.Text(description='📺 תוכנית אהובה:', placeholder='למשל: Breaking Bad',
                   style=style, layout=wide, disabled=True)
fav.add_class('fav-box')
btn = widgets.Button(description='עדכון', icon='save', button_style='success', disabled=True,
                     layout=widgets.Layout(width='140px', margin='10px 0 0 125px'))
msg = widgets.HTML()

def show_student(change):
    sid = change['new']
    msg.value = ''
    if sid is None:
        for w in (first_name, last_name, email, courses, fav): w.value = ''
        link.value = ''; fav.disabled = btn.disabled = True
        return
    s = next(x for x in load_students() if x['id'] == sid)
    first_name.value = s['first_name']
    last_name.value  = s['last_name']
    email.value      = s['email']
    courses.value    = '\n'.join(f'• {c}' for c in s['courses'])
    url = escape(s['interesting_link'])
    link.value = f'<b style="display:inline-block;width:120px">קישור מעניין:</b> <a href="{url}" target="_blank">{url} 🔗</a>'
    fav.value = s.get('favorite_show', '')
    fav.disabled = btn.disabled = False

def on_update(_):
    show = fav.value.strip()
    if not show:
        msg.value = '<span style="color:#d63031">⚠️ נא להזין תוכנית</span>'; return
    s = update_favorite_show(dropdown.value, show)
    msg.value = (f'<span style="color:#00b894;font-weight:600">✅ עודכן! '
                 f'התוכנית האהובה של {escape(s["first_name"])}: {escape(show)}</span>')

dropdown.observe(show_student, names='value')
btn.on_click(on_update)

card = widgets.VBox([title, dropdown, first_name, last_name, email, courses, link, fav, btn, msg])
card.add_class('card')
display(card)

In [5]:
# תא 5 (בדיקה) – הצגת תוכן הקובץ אחרי עדכון
print(json.dumps(load_students(), ensure_ascii=False, indent=2))

[
  {
    "id": 1,
    "first_name": "Muhamad",
    "last_name": "Mansour",
    "email": "muhamad.xxx@e.braude.ac.il",
    "courses": [
      "Automata and Computability",
      "Machine Learning",
      "AI Engineering"
    ],
    "interesting_link": "https://colab.research.google.com",
    "favorite_show": ""
  },
  {
    "id": 2,
    "first_name": "Student2",
    "last_name": "LAST_NAME",
    "email": "student2@e.braude.ac.il",
    "courses": [
      "Course A",
      "Course B"
    ],
    "interesting_link": "https://github.com",
    "favorite_show": ""
  },
  {
    "id": 3,
    "first_name": "Student3",
    "last_name": "LAST_NAME",
    "email": "student3@e.braude.ac.il",
    "courses": [
      "Course A",
      "Course C"
    ],
    "interesting_link": "https://stackoverflow.com",
    "favorite_show": ""
  }
]
